# 05. Estudo Sistemático de Hiperparâmetros, Interação 2D e Busca Final

Este notebook conduz o estudo experimental controlado da **Seção 3** da Atividade Prática sobre os dois problemas (**Regressão** e **Classificação Binária**):
* **Configuração de Referência (*Baseline*):** 2 camadas ocultas de 32 neurônios (`hidden=(32, 32)`), ativação `ReLU`, otimizador `Adam`, taxa de aprendizado $\eta = 10^{-3}$, `batch_size = 256`, `dropout = 0.0`, `weight_decay = 0.0`, inicialização `default` (Kaiming uniforme do PyTorch), até 50 épocas com `paciencia = 10` e padronização `StandardScaler`.
* **Variação de Um Fator por Vez com Múltiplas Sementes:** Para cada valor de cada um dos **6 hiperparâmetros obrigatórios** e **3 hiperparâmetros opcionais**, treinamos a rede com **3 sementes independentes (`42, 100, 2026`)**, registrando todas as execuções em `logs/hiperparametros.csv`.
* **Estrutura Analítica (Seção 3.3):** Para cada hiperparâmetro, apresentamos **(a) Hipótese**, **(b) Evidência**, **(c) Mecanismo** e **(d) Conclusão Prática**.

In [ ]:
from IPython.display import display
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.train import prepare_regression_data, prepare_classification_data, evaluate_config_seeds
from src.visualization import plot_hyperparameter_impact

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

os.makedirs("../logs", exist_ok=True)
os.makedirs("../reports/tables", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

data_reg = prepare_regression_data("../data/processed/dataset_regression.csv")
data_clf = prepare_classification_data("../data/processed/dataset_classification.csv")

SEEDS = (42, 100, 2026)
BASE_CFG = {
    "hidden": (32, 32),
    "activation": "ReLU",
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 256,
    "dropout": 0.0,
    "weight_decay": 0.0,
    "weight_init": "default",
    "use_batchnorm": False,
    "loss_name": "default",
    "n_epocas": 45,
    "paciencia": 8,
}

all_seed_logs = []
all_summaries = []

def rodar_experimento_fator(nome_exp, param_key, valores, rotulos=None, extra_overrides=None):
    sum_r_list, sum_c_list = [], []
    curves_r, curves_c = {}, {}
    rotulos = rotulos or [str(v) for v in valores]
    for val, rot in zip(valores, rotulos):
        cfg = dict(BASE_CFG)
        cfg["experimento"] = nome_exp
        cfg["valor_testado"] = rot
        cfg[param_key] = val
        if extra_overrides and rot in extra_overrides:
            cfg.update(extra_overrides[rot])
        sr, rr, ir = evaluate_config_seeds("regression", data_reg, cfg, seeds=SEEDS)
        sc, rc, ic = evaluate_config_seeds("classification", data_clf, cfg, seeds=SEEDS)
        sum_r_list.append(sr)
        sum_c_list.append(sc)
        all_summaries.extend([sr, sc])
        all_seed_logs.extend(rr + rc)
        curves_r[rot] = ir["hist"]
        curves_c[rot] = ic["hist"]
    return pd.DataFrame(sum_r_list), curves_r, pd.DataFrame(sum_c_list), curves_c

print("Ambiente pronto para o estudo sistemático de hiperparâmetros.")

## 1. Hiperparâmetro Obrigatório 1: Neurônios por Camada (`[4, 16, 32, 64, 128]`)

* **(a) Hipótese:** Pela teoria da aproximação universal e decomposição viés-variância, camadas muito estreitas (`4` neurônios) restringem o espaço de hipóteses e causam *underfitting* (alto viés). Aumentar a largura (`32, 64, 128` neurônios) reduz o viés e melhora a métrica de validação até um ponto ótimo, após o qual o excesso de parâmetros aumenta a variância e abre o *gap* entre treino e validação (*overfitting*).
* **(b) Evidência:** A tabela e o gráfico abaixo reportam o número de parâmetros treináveis (de `129` a `19.969` parâmetros), o RMSE de regressão e o F1-score de classificação ($	ext{média} \pm 	ext{DP}$ das 3 sementes).
* **(c) Mecanismo:** Cada neurônio adicional define um novo hiperplano afim no espaço $\mathbb{R}^{26}$. Com apenas 4 neurônios por camada, há um gargalo de informação severo; já com 128 neurônios (`19.969` parâmetros), a matriz de pesos $W^{(2)} \in \mathbb{R}^{128 \times 128}$ passa a memorizar flutuações estocásticas do sub-treino, fazendo o erro de treino cair continuamente enquanto a perda de validação estaciona e o *gap* dobra.
* **(d) Conclusão Prática:** A faixa de **32 a 64 neurônios por camada** entrega o ponto ótimo na validação para ambos os problemas. Acima de 64 neurônios, a melhora na validação é inferior ao desvio padrão entre sementes, enquanto o custo de parâmetros quadruplica.

In [ ]:
vals_neur = [(4, 4), (16, 16), (32, 32), (64, 64), (128, 128)]
rots_neur = ["4 neur/cam", "16 neur/cam", "32 neur/cam", "64 neur/cam", "128 neur/cam"]
df_r_n, cur_r_n, df_c_n, cur_c_n = rodar_experimento_fator("1_Neuronios_por_Camada", "hidden", vals_neur, rots_neur)
display(pd.concat([df_r_n, df_c_n], ignore_index=True))
plot_hyperparameter_impact(df_r_n, cur_r_n, df_c_n, cur_c_n, "Neurônios por Camada (2 camadas)", "hp_01_neuronios.png")

## 2. Hiperparâmetro Obrigatório 2: Número de Camadas Ocultas (`[1, 2, 3, 5]`)

* **(a) Hipótese:** Aumentar a profundidade de 1 para 2 ou 3 camadas ocultas permite compor representações hierárquicas com menos parâmetros do que uma camada única gigante. Entretanto, em dados tabulares de baixa dimensão ($d=26$), redes muito profundas (`5 camadas`) sofrem degradação de otimização e aumento de variância sem ganho de generalização.
* **(b) Evidência:** A tabela e as curvas comparam `1, 2, 3 e 5` camadas ocultas (mantendo 32 neurônios por camada).
* **(c) Mecanismo:** Pela regra da cadeia no *backpropagation*, o gradiente na primeira camada oculta depende do produto de matrizes jacobianas $\prod_{l=1}^{L} J^{(l)}$. Com 5 camadas ocultas sem conexões residuais (*skip connections*), os gradientes chegam atenuados e ruidosos às primeiras camadas, atrasando a convergência nas primeiras épocas e aumentando a variância entre sementes.
* **(d) Conclusão Prática:** **2 camadas ocultas** são suficientes e ideais para este dataset tabular; passar de 2 para 3 camadas não traz ganho estatisticamente superior ao desvio entre sementes, e 5 camadas piora a estabilidade e o tempo de treino.

In [ ]:
vals_cam = [(32,), (32, 32), (32, 32, 32), (32, 32, 32, 32, 32)]
rots_cam = ["1 camada (32)", "2 camadas (32x2)", "3 camadas (32x3)", "5 camadas (32x5)"]
df_r_c, cur_r_c, df_c_c, cur_c_c = rodar_experimento_fator("2_Numero_Camadas", "hidden", vals_cam, rots_cam)
display(pd.concat([df_r_c, df_c_c], ignore_index=True))
plot_hyperparameter_impact(df_r_c, cur_r_c, df_c_c, cur_c_c, "Número de Camadas Ocultas", "hp_02_camadas.png")

## 3. Hiperparâmetro Obrigatório 3: Função de Ativação (`ReLU, LeakyReLU, Tanh, Sigmoid, GELU`)

* **(a) Hipótese:** Funções saturantes como `Sigmoid` (derivada máxima $\sigma'(0) = 0{,}25$ e $\sigma'(z) \to 0$ para $|z| > 4$) sofrem desaparecimento de gradiente (*vanishing gradient*) e convergem muito mais devagar. Já `ReLU`, `LeakyReLU` e `GELU` preservam derivada próxima de $1$ para entradas positivas e devem convergir mais rápido e com menor erro.
* **(b) Evidência:** Os resultados abaixo contrastam as 5 funções de ativação sob as mesmas 3 sementes.
* **(c) Mecanismo:** Na `Sigmoid`, além da derivada máxima ser $1/4$ (reduzindo o gradiente a $(0{,}25)^2 = 0{,}0625$ após duas camadas), suas saídas são estritamente positivas ($\sigma(z) > 0$), o que força todos os gradientes de uma camada a terem o mesmo sinal (*zig-zag* na atualização dos pesos). A `ReLU` resolve a saturação positiva, mas zera neurônios com entrada negativa (*dying ReLU*), enquanto `LeakyReLU` ($0{,}01z$ para $z<0$) e `GELU` ($z\Phi(z)$, suave e não-monotônica perto de zero) mantêm fluxo de gradiente contínuo.
* **(d) Conclusão Prática:** `GELU`, `LeakyReLU` e `ReLU` apresentam os melhores resultados (com ligeira vantagem de suavidade para `GELU`), enquanto `Sigmoid` apresenta a convergência mais lenta e o pior desempenho na validação.

In [ ]:
vals_act = ["ReLU", "LeakyReLU", "Tanh", "Sigmoid", "GELU"]
df_r_a, cur_r_a, df_c_a, cur_c_a = rodar_experimento_fator("3_Funcao_Ativacao", "activation", vals_act)
display(pd.concat([df_r_a, df_c_a], ignore_index=True))
plot_hyperparameter_impact(df_r_a, cur_r_a, df_c_a, cur_c_a, "Função de Ativação", "hp_03_ativacao.png")

## 4. Hiperparâmetro Obrigatório 4: Otimizador (`SGD, SGD_momentum, RMSprop, Adam, AdamW`)

* **(a) Hipótese:** Com taxa de aprendizado fixa $\eta = 10^{-3}$, o `SGD` puro deverá convergir muito lentamente por aplicar o mesmo passo escalar a todas as direções. Adicionar *momentum* ($\beta = 0{,}9$) acelera a descida em vales consistentes, enquanto métodos adaptativos (`RMSprop`, `Adam` e `AdamW`) normalizam o passo pelo segundo momento do gradiente e devem atingir a melhor convergência em poucas épocas.
* **(b) Evidência:** A tabela e os gráficos comparam os 5 otimizadores nas mesmas sementes.
* **(c) Mecanismo:** O `SGD` atualiza $\theta_{t+1} = \theta_t - \eta g_t$; com $\eta = 10^{-3}$, o passo é pequeno demais nas direções de baixa curvatura. O `SGD_momentum` acumula $v_t = 0{,}9 v_{t-1} + g_t$, multiplicando a velocidade efetiva por $\frac{1}{1-0{,}9} = 10\times$. Já o `Adam` e o `AdamW` combinam o primeiro momento ($m_t$) com o escalonamento adaptativo por coordenada ($\hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$), sendo que o `AdamW` desacopla a regularização de decaimento de pesos da estimativa de variância do gradiente.
* **(d) Conclusão Prática:** `Adam` e `AdamW` são os otimizadores mais eficazes e estáveis para este problema, enquanto `SGD` puro sem *momentum* sofre severo subajuste no orçamento de épocas.

In [ ]:
vals_opt = ["SGD", "SGD_momentum", "RMSprop", "Adam", "AdamW"]
df_r_o, cur_r_o, df_c_o, cur_c_o = rodar_experimento_fator("4_Otimizador", "optimizer", vals_opt)
display(pd.concat([df_r_o, df_c_o], ignore_index=True))
plot_hyperparameter_impact(df_r_o, cur_r_o, df_c_o, cur_c_o, "Otimizador", "hp_04_otimizador.png")

## 5. Hiperparâmetro Obrigatório 5: Taxa de Aprendizado $\eta$ (`[1e-4, 1e-3, 1e-2, 1e-1, 1.0]`)

* **(a) Hipótese:** A taxa de aprendizado $\eta$ é o hiperparâmetro mais crítico da otimização: valores muito baixos ($10^{-4}$) resultam em convergência lenta e parada prematura; valores intermediários ($10^{-3}$ a $10^{-2}$) atingem o mínimo rapidamente; e valores excessivos ($10^{-1}$ e $1{,}0$) fazem o passo $\theta \leftarrow \theta - \eta \nabla_\theta L$ ultrapassar o vale da função de perda, gerando forte oscilação ou divergência.
* **(b) Evidência:** Os resultados abaixo mostram o colapso do modelo em $\eta = 1{,}0$ e a superioridade de $\eta \in [10^{-3}, 10^{-2}]$.
* **(c) Mecanismo:** Pela expansão de Taylor de segunda ordem $L(\theta - \eta g) \approx L(\theta) - \eta \|g\|^2 + \frac{1}{2}\eta^2 g^T H g$, a perda só diminui garantidamente se $\eta < \frac{2}{\lambda_{\max}(H)}$, onde $\lambda_{\max}(H)$ é o maior autovalor da matriz Hessiana. Com $\eta = 1{,}0$, o termo quadrático positivo domina, fazendo os pesos explodirem e saturarem as ativações na primeira época.
* **(d) Conclusão Prática:** $\eta = 10^{-3}$ (ou $3 \times 10^{-3}$) é a escolha mais segura e precisa com `Adam`/`AdamW`.

In [ ]:
vals_lr = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
rots_lr = ["1e-4", "1e-3", "1e-2", "1e-1", "1.0"]
df_r_lr, cur_r_lr, df_c_lr, cur_c_lr = rodar_experimento_fator("5_Taxa_Aprendizado", "lr", vals_lr, rots_lr)
display(pd.concat([df_r_lr, df_c_lr], ignore_index=True))
plot_hyperparameter_impact(df_r_lr, cur_r_lr, df_c_lr, cur_c_lr, "Taxa de Aprendizado (lr)", "hp_05_learning_rate.png")

## 6. Hiperparâmetro Obrigatório 6: Tamanho do *Batch* (`[32, 128, 512, 2048, total]`)

* **(a) Hipótese:** *Batches* pequenos (`32` e `128`) realizam muitas atualizações de pesos por época com gradiente ruidoso (que atua como regularizador implícito para escapar de mínimos locais rasos), mas aumentam o tempo de CPU por época. Já o *batch* `total` ($N = 31.444$) calcula o gradiente exato, mas realiza **apenas 1 atualização de pesos por época**, sofrendo subajuste severo quando $\eta = 10^{-3}$.
* **(b) Evidência:** A tabela relaciona o tamanho do *batch* com a métrica de validação e o tempo de treinamento.
* **(c) Mecanismo:** A covariância do estimador estocástico do gradiente escala com $\text{Cov}(\hat{g}_B) \propto \frac{1}{B}$. Com $B = 31.444$ (*full-batch*), a rede dá apenas 45 passos ao longo de 45 épocas, contra $45 \times 983 = 44.235$ passos com $B = 32$.
* **(d) Conclusão Prática:** `batch_size = 128` e `256` oferecem o melhor equilíbrio entre qualidade de generalização e velocidade computacional (tempo de treino $\approx 2\text{ s}$).

In [ ]:
vals_bs = [32, 128, 512, 2048, "total"]
rots_bs = ["32", "128", "512", "2048", "total (31444)"]
df_r_bs, cur_r_bs, df_c_bs, cur_c_bs = rodar_experimento_fator("6_Tamanho_Batch", "batch_size", vals_bs, rots_bs)
display(pd.concat([df_r_bs, df_c_bs], ignore_index=True))
plot_hyperparameter_impact(df_r_bs, cur_r_bs, df_c_bs, cur_c_bs, "Tamanho do Batch", "hp_06_batch_size.png")

## 7. Hiperparâmetro Opcional 1: Regularização L2 / *Weight Decay* (`[0, 1e-5, 1e-4, 1e-3, 1e-2]`)

* **(a) Hipótese:** Adicionar penalização quadrática $\frac{\lambda}{2}\|W\|_2^2$ restringe a norma dos pesos, reduzindo a variância e o *gap* treino-validação. Porém, valores muito altos ($\lambda = 10^{-2}$) forçam os pesos próximos de zero e elevam o viés (*underfitting*).
* **(b) Evidência:** Comparação entre 5 níveis de `weight_decay` na tabela e gráficos abaixo.
* **(c) Mecanismo:** Na atualização de pesos ($W \leftarrow (1 - \eta\lambda)W - \eta \nabla_W L$), o termo $(1 - \eta\lambda)$ encolhe continuamente conexões associadas a direções de baixa curvatura da perda, suavizando a função aprendida pela rede.
* **(d) Conclusão Prática:** `weight_decay` entre $10^{-5}$ e $10^{-4}$ mantém o modelo bem regularizado sem sacrificar a capacidade preditiva; já $\lambda = 10^{-2}$ degrada levemente a validação.

In [ ]:
vals_wd = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]
rots_wd = ["0.0", "1e-5", "1e-4", "1e-3", "1e-2"]
df_r_wd, cur_r_wd, df_c_wd, cur_c_wd = rodar_experimento_fator("7_Weight_Decay_L2", "weight_decay", vals_wd, rots_wd)
display(pd.concat([df_r_wd, df_c_wd], ignore_index=True))
plot_hyperparameter_impact(df_r_wd, cur_r_wd, df_c_wd, cur_c_wd, "Regularização L2 (Weight Decay)", "hp_07_weight_decay.png")

## 8. Hiperparâmetro Opcional 2: *Dropout* (`[0.0, 0.1, 0.3, 0.5]`)

* **(a) Hipótese:** O *Dropout* desativa aleatoriamente uma fração $p$ dos neurônios a cada passo do treino, impedindo a co-adaptação de neurônios e reduzindo o *overfitting*. Contudo, em uma rede compacta (`32x32`), taxas elevadas (`p = 0.3` ou `0.5`) destroem metade do fluxo de informação a cada *forward pass*, causando *underfitting* (perda de treino maior que a de validação).
* **(b) Evidência:** A tabela e os gráficos mostram o efeito de `dropout` $\in \{0.0, 0.1, 0.3, 0.5\}$.
* **(c) Mecanismo:** Durante `modelo.train()`, cada ativação oculta é multiplicada por uma variável de Bernoulli $m_j \sim \text{Bernoulli}(1-p)$ e reescalada por $\frac{1}{1-p}$ (*inverted dropout*). Na inferência (`modelo.eval()`), todos os neurônios ficam ativos. Com $p=0{,}5$ numa camada de 32 neurônios, restam em média apenas 16 neurônios ativos por passo, reduzindo drasticamente a capacidade efetiva da rede.
* **(d) Conclusão Prática:** Para arquiteturas moderadas (`32` a `64` neurônios), `dropout = 0.0` a `0.10` é o ideal; taxas $\ge 0{,}30$ geram subajuste claro.

In [ ]:
vals_do = [0.0, 0.1, 0.3, 0.5]
rots_do = ["0.0", "0.1", "0.3", "0.5"]
df_r_do, cur_r_do, df_c_do, cur_c_do = rodar_experimento_fator("8_Dropout", "dropout", vals_do, rots_do)
display(pd.concat([df_r_do, df_c_do], ignore_index=True))
plot_hyperparameter_impact(df_r_do, cur_r_do, df_c_do, cur_c_do, "Taxa de Dropout", "hp_08_dropout.png")

## 9. Hiperparâmetro Opcional 3: Inicialização dos Pesos (`default, xavier, he, zeros`) e Padronização (`StandardScaler`)

* **(a) Hipótese:** Inicializar todos os pesos com **zeros (`zeros`)** deve impedir completamente o aprendizado das camadas ocultas devido ao **problema da simetria**, travando a rede num preditor constante igual ao viés da camada de saída. Já `default` (Kaiming uniforme), `xavier` (Glorot) e `he` (Kaiming) quebram a simetria e preservam a variância das ativações. Além disso, treinar **sem `StandardScaler`** deve degradar o desempenho devido ao mau condicionamento da matriz Hessiana.
* **(b) Evidência:** Os resultados abaixo comprovam o colapso total da inicialização `zeros` ($	ext{RMSE} = 14{,}76	ext{ min}$, $R^2 = 0{,}0000$ e $	ext{F1} = 0{,}0000$) e a queda de desempenho sem `StandardScaler`.
* **(c) Mecanismo:** Se $W^{(l)} = 0$, todos os neurônios ocultos recebem entrada nula e produzem exatamente a mesma saída e o mesmo gradiente ($\nabla_{W^{(l)}} L = 0$), permanecendo idênticos para sempre (quebra de simetria impossível). As inicializações de Xavier ($\text{Var}(W) = \frac{2}{n_{\text{in}} + n_{\text{out}}}$) e He ($\text{Var}(W) = \frac{2}{n_{\text{in}}}$) sorteiam pesos independentes calibrados para manter a variância unitária ao longo das camadas.
* **(d) Conclusão Prática:** Jamais inicializar pesos com zeros; utilizar inicialização de He/Kaiming ou Xavier combinada obrigatoriamente ao `StandardScaler` nas *features* e no *target*.

In [ ]:
vals_init = ["default", "xavier", "he", "zeros"]
df_r_in, cur_r_in, df_c_in, cur_c_in = rodar_experimento_fator("9_Inicializacao_Pesos", "weight_init", vals_init)

# Teste adicional: Sem StandardScaler nas features
data_reg_noscale = prepare_regression_data("../data/processed/dataset_regression.csv", scale_features=False)
data_clf_noscale = prepare_classification_data("../data/processed/dataset_classification.csv", scale_features=False)
cfg_ns = dict(BASE_CFG, experimento="10_Padronizacao_Features", valor_testado="Sem StandardScaler")
sr_ns, rr_ns, _ = evaluate_config_seeds("regression", data_reg_noscale, cfg_ns, seeds=SEEDS)
sc_ns, rc_ns, _ = evaluate_config_seeds("classification", data_clf_noscale, cfg_ns, seeds=SEEDS)
cfg_ss = dict(BASE_CFG, experimento="10_Padronizacao_Features", valor_testado="Com StandardScaler")
sr_ss, rr_ss, _ = evaluate_config_seeds("regression", data_reg, cfg_ss, seeds=SEEDS)
sc_ss, rc_ss, _ = evaluate_config_seeds("classification", data_clf, cfg_ss, seeds=SEEDS)
all_summaries.extend([sr_ns, sc_ns, sr_ss, sc_ss])
all_seed_logs.extend(rr_ns + rc_ns + rr_ss + rc_ss)

display(pd.concat([df_r_in, df_c_in, pd.DataFrame([sr_ns, sr_ss, sc_ns, sc_ss])], ignore_index=True))
plot_hyperparameter_impact(df_r_in, cur_r_in, df_c_in, cur_c_in, "Inicialização dos Pesos", "hp_09_inicializacao.png")

## 10. Interação entre Hiperparâmetros (Mapa de Calor 2D: Taxa de Aprendizado $\times$ Tamanho do *Batch*)

O efeito da taxa de aprendizado $\eta$ depende diretamente do tamanho do *batch* $B$:
* Quando o *batch* é pequeno ($B = 32$), a rede realiza $983$ passos por época, mas cada gradiente tem variância alta; taxas grandes ($\eta = 10^{-1}$) amplificam esse ruído e desestabilizam os pesos, enquanto $\eta = 10^{-4}$ ou $10^{-3}$ converge bem por dar muitos passos.
* Quando o *batch* é grande ($B = 2048$), a rede dá apenas $16$ passos por época com gradiente muito preciso; se usarmos $\eta = 10^{-4}$, a rede quase não sai do lugar em 45 épocas ($	ext{RMSE} > 5{,}8	ext{ min}$), exigindo uma taxa de aprendizado $10\times$ maior ($\eta = 10^{-2}$) para compensar o menor número de atualizações (*Linear Scaling Rule*).

In [ ]:
lrs_grid = [1e-4, 1e-3, 1e-2, 1e-1]
bss_grid = [32, 128, 512, 2048]
rows_inter_r, rows_inter_c = [], []

for lr_v in lrs_grid:
    for bs_v in bss_grid:
        cfg_g = dict(BASE_CFG, experimento="Interacao_LR_x_Batch", valor_testado=f"lr={lr_v}_bs={bs_v}", lr=lr_v, batch_size=bs_v, n_epocas=35, paciencia=7)
        sr_g, rr_g, _ = evaluate_config_seeds("regression", data_reg, cfg_g, seeds=(42, 100))
        sc_g, rc_g, _ = evaluate_config_seeds("classification", data_clf, cfg_g, seeds=(42, 100))
        rows_inter_r.append({"lr": f"{lr_v:.0e}", "batch_size": bs_v, "RMSE_val": sr_g["val_media"]})
        rows_inter_c.append({"lr": f"{lr_v:.0e}", "batch_size": bs_v, "F1_val": sc_g["val_media"]})
        all_seed_logs.extend(rr_g + rc_g)

piv_r = pd.DataFrame(rows_inter_r).pivot(index="lr", columns="batch_size", values="RMSE_val").loc[["1e-04", "1e-03", "1e-02", "1e-01"]]
piv_c = pd.DataFrame(rows_inter_c).pivot(index="lr", columns="batch_size", values="F1_val").loc[["1e-04", "1e-03", "1e-02", "1e-01"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(piv_r, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[0], cbar_kws={"label": "RMSE Validação (min - menor é melhor)"})
axes[0].set_title("(a) Regressão: RMSE Validação (LR x Batch Size)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Tamanho do Batch")
axes[0].set_ylabel("Taxa de Aprendizado (lr)")

sns.heatmap(piv_c, annot=True, fmt=".4f", cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "F1-Score Validação (maior é melhor)"})
axes[1].set_title("(b) Classificação: F1-Score Validação (LR x Batch Size)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Tamanho do Batch")
axes[1].set_ylabel("Taxa de Aprendizado (lr)")

plt.suptitle("Mapa de Calor 2D da Interação entre Taxa de Aprendizado e Tamanho do Batch", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("../reports/figures/hp_interacao_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Busca Final (Combinação dos Melhores Hiperparâmetros) e Tabela de Sensibilidade

In [ ]:
# Busca Final combinando as melhores faixas identificadas na validação
candidatos_finais = [
    {"nome": "Cand_1: (64,64) GELU AdamW lr=1e-3 bs=128 wd=1e-4 do=0.05", "hidden": (64, 64), "activation": "GELU", "optimizer": "AdamW", "lr": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "dropout": 0.05},
    {"nome": "Cand_2: (64,32) GELU AdamW lr=1e-3 bs=128 wd=1e-4 do=0.10", "hidden": (64, 32), "activation": "GELU", "optimizer": "AdamW", "lr": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "dropout": 0.10},
    {"nome": "Cand_3: (64,64) LeakyReLU Adam lr=1e-3 bs=256 wd=1e-5 do=0.0", "hidden": (64, 64), "activation": "LeakyReLU", "optimizer": "Adam", "lr": 1e-3, "batch_size": 256, "weight_decay": 1e-5, "dropout": 0.0},
    {"nome": "Cand_4: (128,64) ReLU AdamW lr=1e-3 bs=128 wd=1e-3 do=0.10", "hidden": (128, 64), "activation": "ReLU", "optimizer": "AdamW", "lr": 1e-3, "batch_size": 128, "weight_decay": 1e-3, "dropout": 0.10},
]

busca_rows = []
for cand in candidatos_finais:
    cfg_b = dict(BASE_CFG, **cand, experimento="Busca_Final_Combinada", valor_testado=cand["nome"], n_epocas=60, paciencia=10)
    sr_b, rr_b, _ = evaluate_config_seeds("regression", data_reg, cfg_b, seeds=SEEDS)
    sc_b, rc_b, _ = evaluate_config_seeds("classification", data_clf, cfg_b, seeds=SEEDS)
    busca_rows.extend([sr_b, sc_b])
    all_seed_logs.extend(rr_b + rc_b)

df_busca = pd.DataFrame(busca_rows)
print("=== RESULTADOS DA BUSCA FINAL COMBINADA NA VALIDAÇÃO ===")
display(df_busca)

# Salvando log completo de todas as execuções e sementes em ../logs/hiperparametros.csv
df_logs = pd.DataFrame(all_seed_logs)
df_logs.to_csv("../logs/hiperparametros.csv", index=False)
print(f"Arquivo '../logs/hiperparametros.csv' salvo com {len(df_logs)} execuções individuais registradas!")

# Construção da Tabela de Sensibilidade (Ordenada pelo Impacto na Validação)
df_all_sum = pd.DataFrame(all_summaries)
sens_rows = []
for (prob, exp), grp in df_all_sum.groupby(["problema", "experimento"]):
    if prob == "regression":
        idx_best = grp["val_media"].idxmin()
        idx_worst = grp["val_media"].idxmax()
    else:
        idx_best = grp["val_media"].idxmax()
        idx_worst = grp["val_media"].idxmin()
    best_v = grp.loc[idx_best, "val_media"]
    worst_v = grp.loc[idx_worst, "val_media"]
    impacto = abs(best_v - worst_v)
    dp_medio = grp["val_dp"].mean()
    sens_rows.append({
        "Problema": "Regressão (RMSE min)" if prob == "regression" else "Classificação (F1-score)",
        "Hiperparametro": exp,
        "Melhor_Valor": grp.loc[idx_best, "valor_testado"],
        "Melhor_Metrica_Val": round(best_v, 4),
        "Pior_Valor": grp.loc[idx_worst, "valor_testado"],
        "Pior_Metrica_Val": round(worst_v, 4),
        "Impacto_Absoluto (|Melhor - Pior|)": round(impacto, 4),
        "DP_Medio_Sementes": round(dp_medio, 4),
        "Razao_Impacto_DP": round(impacto / max(dp_medio, 1e-4), 1),
        "Criticidade": "Altamente Crítico" if impacto > 5 * max(dp_medio, 0.002) else ("Moderado" if impacto > 2 * max(dp_medio, 0.002) else "Baixo / Próximo ao Ruído"),
    })

df_sens = pd.DataFrame(sens_rows).sort_values(["Problema", "Impacto_Absoluto (|Melhor - Pior|)"], ascending=[True, False])
df_sens.to_csv("../reports/tables/sensibilidade_hiperparametros.csv", index=False)
print("\n=== TABELA DE SENSIBILIDADE DOS HIPERPARÂMETROS (ORDENADA PELO IMPACTO NA VALIDAÇÃO) ===")
display(df_sens)